In [ ]:
"""
This script processes the Output Based Pricing System (OBPS) as an addition to the fixed cost of selected technologies.
The technologies and relations are adjustable and can be modified.
In this version, the processing is based on 'CO2_Emission', which represents emissions of CO2 (measured as carbon, ton C/kWyr).
The script calculates OBPS for two scenarios: Baseline and NetZero.
"""

import pandas as pd

file_path = 'data_CA_provincial_reference_v1.xlsx'

# Read the required data
data_relation = pd.read_excel(file_path, sheet_name='relation_activity')

# Define the technologies and relations
#conditions_1 = (data_relation['technology'].isin(['oil_extr_1', 'coal_ppl','oil_extr_2', 'foil_ppl'])) & (data_relation['relation'] == 'CO2_cc') & (data_relation['node_rel'] == 'Alberta')
conditions_2 = (data_relation['technology'].isin(['coal_extr','oil_extr_1', 'coal_ppl','oil_extr_2', 'foil_ppl','gas_extr_1'])) & (data_relation['relation'] == 'CO2_Emission') & (data_relation['node_rel'] == 'Alberta')

# Filter the data based on the conditions
filtered_data = data_relation[conditions_2]


# Select the relevant columns
result = filtered_data[['technology', 'value', 'node_rel', 'year_act']]

# Save the result to a new Excel file
result.to_csv('filtered_data.csv', index=False)




In [ ]:
result.loc[:,'Emission Intensity Reduction_baseline'] = 0.0
for index, row in result.iterrows():
    if 2020 < row['year_act'] < 2055:
        previous_year_value = result.loc[(result['technology'] == row['technology']) & (result['year_act'] == row['year_act'] - 5), 'Emission Intensity Reduction_baseline'].values
        if len(previous_year_value) > 0:
            result.loc[index, 'Emission Intensity Reduction_baseline'] += previous_year_value[0] + 3.33/100

result.to_csv('filtered_data.csv', index=False)

In [ ]:
#Emission intensity standard_Baseline Canada Scenario 
result.loc[:, 'Emission intensity standard_Baseline Canada Scenario'] = 0.0

for index, row in result.iterrows():
    if row['year_act'] > 2020:
        value_cal = result.loc[(result['technology'] == row['technology'])& (result['year_act']==2015), 'value'].values
        result.loc[index, 'Emission intensity standard_Baseline Canada Scenario']= (row['value']-row['Emission Intensity Reduction_baseline']* value_cal)

result.to_csv('filtered_data.csv', index=False)

In [ ]:
#Payable emission intensity_Baseline scenario
#based on documentation: CO2_Emission is Emissions of CO2 (weight as carbon, ton C/kWyr)
result.loc[:, 'Payable emission intensity_Baseline scenario'] = 0.0

for index, row in result.iterrows():
    if row['year_act'] > 2020:
        result.loc[index, 'Payable emission intensity_Baseline scenario']= row['value']-row['Emission intensity standard_Baseline Canada Scenario']
        
# Save the updated data back to the same CSV file
result.to_csv('filtered_data.csv', index=False)

In [ ]:
#carbon price ($ CAD/tonne CO2), CAD 1=US 0.73
rate= 0.73
#carbon cost_Baseline Canada scenario ($ USD/tonne CO2)*(ton C/kWyr) so the final unit is (USD/kwyr)
result.loc[:, 'carbon cost_Baseline Canada scenario'] =0.0

for index, row in result.iterrows():
    if row['year_act']== 2025:
        result.loc[index,'carbon cost_Baseline Canada scenario']= row['Payable emission intensity_Baseline scenario']*rate*125
    elif row['year_act']> 2025:
        result.loc[index,'carbon cost_Baseline Canada scenario']= row['Payable emission intensity_Baseline scenario']*rate*170


result.to_csv('filtered_data.csv', index=False)


In [ ]:
#add to fix_cost (USD/kw)
data_fixcost = pd.read_excel(file_path, sheet_name='fix_cost')


# Merge the dataframes on 'technology' and 'year' columns
merged_data = pd.merge(data_fixcost, result[['technology', 'year_act', 'carbon cost_Baseline Canada scenario']],
                       left_on=['technology', 'year_act'],
                       right_on=['technology', 'year_act'],
                       how='left')

# Update the 'value' column in fix_cost by adding the 'payable cost' from filtered_data
merged_data['value'] = merged_data['value'] + merged_data['carbon cost_Baseline Canada scenario'].fillna(0)


merged_data.drop(columns=['carbon cost_Baseline Canada scenario'], inplace=True)

# Save the updated fix_cost data to a new CSV file
merged_data.to_csv('updated_obps_baseline_fix_cost.csv', index=False)


In [ ]:
#calculation for NetZero scenario
result = filtered_data[['technology', 'value', 'node_rel', 'year_act']]
#Emission Intensity Reduction_Net Zero Scenario
#A 2% yearly tightening rate on emissions intensity is applied from 2023 to 2050

result.loc[:,'Emission Intensity Reduction_Net Zero Scenario']=0.0

for index, row in result.iterrows():
    if 2055> row['year_act'] >2020:
        result.loc [index, 'Emission Intensity Reduction_Net Zero Scenario'] = (row['year_act']-2022)*0.02
result.to_csv('new_NZero.csv', index=False)



In [ ]:
#Emission intensity standard_Net zero
result.loc[:,'Emission intensity standard_Net zero']=0.0
for index, row in result.iterrows():
    if 2055> row['year_act'] >2020:
        result.loc[index, 'Emission intensity standard_Net zero']= row['value']*(1-row['Emission Intensity Reduction_Net Zero Scenario'])
result.to_csv('new_NZero.csv', index=False)

In [ ]:
#Payable emission intensity_Net zero
result.loc[:,'Payable emission intensity_Net zero']=0.0
for index, row in result.iterrows():
    if 2055> row['year_act'] >2020:
        result.loc[index,'Payable emission intensity_Net zero']= (row['value']-row['Emission intensity standard_Net zero'])
result.to_csv('new_NZero.csv', index=False)

In [ ]:
#carbon cost_NetZero
rate= 0.73
#carbon cost_Baseline Canada scenario ($ USD/tonne CO2)*(ton C/kWyr) so the final unit is (USD/kwyr)
result.loc[:, 'carbon cost_Net zero scenario'] =0.0

for index, row in result.iterrows():
    if row['year_act']== 2025:
        result.loc[index,'carbon cost_Net zero scenario']= row['Payable emission intensity_Net zero']*rate*125
    elif row['year_act']> 2025:
        result.loc[index,'carbon cost_Net zero scenario']= row['Payable emission intensity_Net zero']*rate*170

result.to_csv('new_NZero.csv', index=False)

In [ ]:
#add to fix_cost (USD/kw)
data_fixcost = pd.read_excel(file_path, sheet_name='fix_cost')

# Merge the dataframes on 'technology' and 'year' columns
merged_data = pd.merge(data_fixcost, result[['technology', 'year_act', 'carbon cost_Net zero scenario']],
                    left_on=['technology', 'year_act'],
                    right_on=['technology', 'year_act'],
                    how='left')

# Update the 'value' column in fix_cost by adding the 'payable cost' from filtered_data
merged_data['value'] = merged_data['value'] + merged_data['carbon cost_Net zero scenario'].fillna(0)

merged_data.drop(columns=['carbon cost_Net zero scenario'], inplace=True)

merged_data.to_csv('updated_obps_NetZero_fix_cost.csv', index=False)